# Game of 24 — Tree of Thoughts Paper Replication
**CS 4782 Final Project — Cornell University**

Direct replication of **Yao et al. NeurIPS 2023, Table 2**:

| Method | Paper (GPT-4) | Ours (Qwen2.5-7B) |
|---|---|---|
| IO | 7.3% | ? |
| CoT | 4.0% | ? |
| CoT-SC (k=100) | 9.0% | ? |
| **ToT-BFS (b=5)** | **74%** | **?** |

**Task**: Use +, -, *, / on 4 numbers to reach 24 (each number used exactly once).  
**Tree structure**: 3 thought steps (4 numbers → 3 → 2 → 1).  
**Evaluator**: Pure LLM — sure / maybe / impossible (×3 samples, summed).  
**Dataset**: 100 hardest puzzles from 4nums.com (rank 901–1000).

> Runtime → Change runtime type → **T4 GPU** for faster inference.

## Step 1: Install Dependencies

In [ ]:
!pip install -q openai datasets matplotlib numpy tqdm

## Step 2: Install Ollama + Pull Model
Same setup as the debugging notebook. Skip if already done this session.

In [ ]:
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time
proc = subprocess.Popen(['ollama', 'serve'],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
print("Ollama server started.")

In [ ]:
!ollama pull qwen2.5:7b
!ollama list


## Step 3: Clone Repo

In [ ]:
import os, subprocess, sys

os.chdir('/content')
repo_dir = '/content/Tree-of-Thoughts-Debugger'
if not os.path.exists(repo_dir):
    subprocess.run(
        ['git', 'clone', 'https://github.com/Datboy0127/Tree-of-Thoughts-Debugger.git'],
        cwd='/content', check=True
    )
    print('Repository cloned.')
else:
    result = subprocess.run(['git', 'pull'], cwd=repo_dir, capture_output=True, text=True)
    print('Repository already exists — pulled latest:', result.stdout.strip() or 'up to date')

sys.path.insert(0, f'{repo_dir}/code')
os.chdir(repo_dir)
print('Working directory:', os.getcwd())


## Step 4: (Optional) Download the Paper's Exact Test Set
The paper uses problems 901-1000 from 4nums.com.  
Run this cell to download the CSV. If it fails, the built-in list of 100 hard puzzles is used.

In [ ]:
import urllib.request, os

csv_url = "https://raw.githubusercontent.com/princeton-nlp/tree-of-thought-llm/master/src/tot/data/24/24.csv"
os.makedirs("data", exist_ok=True)

try:
    urllib.request.urlretrieve(csv_url, "data/24.csv")
    print("Downloaded data/24.csv")
    GAME24_CSV = "data/24.csv"
except Exception as e:
    print(f"Download failed ({e}). Using built-in hard puzzle set.")
    GAME24_CSV = None

## Step 5: Load Puzzles

In [ ]:
from data_loader import load_game24

G24_N = 100   # set to 20 for a quick test run

puzzles = load_game24(n=G24_N, csv_path=GAME24_CSV, seed=42)
print(f"Loaded {len(puzzles)} puzzles")
print("Sample (first 5):", puzzles[:5])

## Step 6: Run All Methods
**Order**: IO → CoT → CoT-SC → ToT-BFS → ToT-DFS  
**Estimated time** (100 puzzles, T4): IO ~5min, CoT ~10min, ToT-BFS ~60-90min

In [ ]:
from llm_client import LLMClient
from game24_solver import (
    Game24Solver, Game24IOBaseline, Game24CoTBaseline, Game24CoTSCBaseline,
    Game24MCTSSolver,
    run_game24, compute_game24_metrics, print_game24_table,
)

llm = LLMClient()
os.makedirs('results', exist_ok=True)
g24 = {}


In [ ]:
# ── IO (paper: 7.3% on GPT-4) ────────────────────────────────────────────────
print("── IO ──")
g24["io"] = run_game24(puzzles, Game24IOBaseline(llm), "io", verbose=False)
m = compute_game24_metrics(g24["io"])
print(f"  IO: {m['success_rate']:.1%}  (paper GPT-4: 7.3%)")

In [ ]:
# ── CoT (paper: 4.0% on GPT-4) ───────────────────────────────────────────────
print("── CoT ──")
g24["cot"] = run_game24(puzzles, Game24CoTBaseline(llm), "cot", verbose=False)
m = compute_game24_metrics(g24["cot"])
print(f"  CoT: {m['success_rate']:.1%}  (paper GPT-4: 4.0%)")

In [ ]:
# ── CoT-SC n=5 (paper uses n=100 for 9.0%) ───────────────────────────────────
print("── CoT-SC (n=5) ──")
g24["cot-sc-5"] = run_game24(puzzles, Game24CoTSCBaseline(llm, n_samples=5), "cot-sc-5", verbose=False)
m = compute_game24_metrics(g24["cot-sc-5"])
print(f"  CoT-SC(5): {m['success_rate']:.1%}  (paper GPT-4, n=100: 9.0%)")

In [ ]:
# ── ToT-BFS b=5 — MAIN PAPER RESULT (paper: 74% on GPT-4) ────────────────────
# Algorithm 1: beam width b=5, 3 thought steps
# Each puzzle: propose prompt × 3 depths + value prompt × 3×n_eval evaluations
# Early stopping: returns as soon as beam contains a state = 24
print("── ToT-BFS (b=5) — this is the main paper result ──")
g24["tot-bfs-b5"] = run_game24(
    puzzles,
    Game24Solver(llm, b=5, n_eval=3, search="bfs"),
    "tot-bfs-b5",
    verbose=True,
)
m = compute_game24_metrics(g24["tot-bfs-b5"])
print(f"\nToT-BFS(b=5): {m['success_rate']:.1%}  avg_tokens={m['avg_tokens']:.0f}  (paper GPT-4: 74.0%)")

In [ ]:
# ── ToT-DFS (Algorithm 2: prune impossible, backtrack on dead ends) ───────────
print("── ToT-DFS ──")
g24["tot-dfs-b5"] = run_game24(
    puzzles,
    Game24Solver(llm, b=5, n_eval=3, search="dfs"),
    "tot-dfs-b5",
    verbose=False,
)
m = compute_game24_metrics(g24["tot-dfs-b5"])
print(f"  ToT-DFS: {m['success_rate']:.1%}  avg_tokens={m['avg_tokens']:.0f}")

In [ ]:
# ── MCTS (novel extension — UCB1 over arithmetic states, random rollouts) ─────
# Selection: UCB1 picks most promising intermediate state.
# Expansion: LLM propose prompt generates b candidate next steps.
# Rollout: pure math (no LLM) — greedy win check then random.
# Early stopping: halts as soon as any simulation finds 24.
print('── MCTS (b=5, s=20) ──')
g24['mcts-b5-s20'] = run_game24(
    puzzles,
    Game24MCTSSolver(llm, b=5, n_simulations=20),
    'mcts-b5-s20',
    verbose=True,
)
m = compute_game24_metrics(g24['mcts-b5-s20'])
print(f"  success={m['success_rate']:.1%}  avg_tokens={m['avg_tokens']:.0f}\n")


## Step 7: Results Table

In [ ]:
print_game24_table(g24)

# Save all results
for label, results in g24.items():
    with open(f"results/game24_{label}.json", "w") as f:
        json.dump([r.__dict__ for r in results], f, indent=2)
print("\nSaved to results/game24_*.json")

## Step 8: Comparison Plot — Ours vs Paper

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: our results ─────────────────────────────────────────────────────────
our_labels = list(g24.keys())
our_rates  = [compute_game24_metrics(g24[l])["success_rate"] * 100 for l in our_labels]
cols1 = ["#9E9E9E" if l in ("io", "cot") or "sc" in l else "#2196F3" for l in our_labels]
b1 = axes[0].bar(np.arange(len(our_labels)), our_rates, color=cols1, alpha=0.85)
axes[0].set_title(f"Our Results (Qwen2.5-7B, n={len(puzzles)})", fontsize=13)
axes[0].set_ylabel("Success Rate (%)", fontsize=12)
axes[0].set_xticks(np.arange(len(our_labels)))
axes[0].set_xticklabels(our_labels, rotation=20, ha="right", fontsize=10)
axes[0].set_ylim(0, 85)
axes[0].bar_label(b1, fmt="%.1f%%", padding=3, fontsize=9)
axes[0].grid(axis="y", alpha=0.3)
axes[0].spines["top"].set_visible(False)
axes[0].spines["right"].set_visible(False)

# ── Right: paper (GPT-4) ──────────────────────────────────────────────────────
paper_labels = ["IO", "CoT", "CoT-SC\n(k=100)", "ToT\n(b=1)", "ToT\n(b=5)"]
paper_rates  = [7.3, 4.0, 9.0, 45.0, 74.0]
cols2 = ["#9E9E9E","#9E9E9E","#9E9E9E","#FF9800","#4CAF50"]
b2 = axes[1].bar(np.arange(len(paper_labels)), paper_rates, color=cols2, alpha=0.85)
axes[1].set_title("Paper Results (GPT-4, Yao et al. NeurIPS 2023)", fontsize=13)
axes[1].set_ylabel("Success Rate (%)", fontsize=12)
axes[1].set_xticks(np.arange(len(paper_labels)))
axes[1].set_xticklabels(paper_labels, fontsize=10)
axes[1].set_ylim(0, 85)
axes[1].bar_label(b2, fmt="%.1f%%", padding=3, fontsize=9)
axes[1].grid(axis="y", alpha=0.3)
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)

fig.suptitle("Game of 24 — ToT Paper Replication (Table 2)", fontsize=15, fontweight="bold")
fig.tight_layout()
plt.savefig("results/game24_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → results/game24_comparison.png")

## Step 9: Show Solved Examples and Error Analysis

In [ ]:
from game24_solver import verify_thought_path

solved   = [r for r in g24.get("tot-bfs-b5", []) if r.success]
unsolved = [r for r in g24.get("tot-bfs-b5", []) if not r.success]

print(f"Solved: {len(solved)}/{len(g24.get('tot-bfs-b5', []))}\n")

print("── First 5 solved puzzles ──")
for r in solved[:5]:
    print(f"  {r.puzzle}")
    for t in r.thoughts:
        print(f"    {t}")
    print(f"  → {r.answer}")
    print()

print(f"── First 5 unsolved puzzles (LLM could not find path) ──")
for r in unsolved[:5]:
    print(f"  {r.puzzle}  (tokens={r.total_tokens})")

## Step 10: Download Results

In [ ]:
import shutil

shutil.make_archive('/content/game24_results', 'zip', 'results')
print("Zipped → /content/game24_results.zip")

try:
    from google.colab import files
    files.download('/content/game24_results.zip')
except ImportError:
    print("Not on Colab — find the zip at /content/game24_results.zip")